# Kalman Historical V3.2 — Corrected Portfolio Validation

V3.1 검토에서 확인된 portfolio/risk 계산 이슈를 교정한 검증 버전입니다.

- True initial equity = 1,000,000
- Monthly rebalance only; sleeve weights drift between rebalances
- Portfolio overlay turnover + 10 bps traded-notional cost
- Calendar-daily annualization = 365 consistently
- Max Sharpe: 365-day inputs + MinVol/InverseVol fallback
- vectorbt: next-open signal fills + close max-hold exits + target-percent sizing
- Overall / Growth / Balanced / Defensive champions separated
- Unit tests must pass before the experiment runs

**Safety:** RESEARCH_ONLY / Toss OFF / Neon write OFF / LIVE OFF


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "7c440fc6d3d0dd1cc2f485dadcb6db03259caba7"
SOURCE_BRANCH = "feature/historical-v3-2-portfolio-validation-20260913"
V1_RUN_TAG = "20260913_042850"
V3_CANDIDATE_TAG = "20260913_return_regime_v3_001"
V32_TAG = "20260913_v3_2_portfolio_validation_001"

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')


def run(cmd, *, cwd=None):
    args = [str(x) for x in cmd]
    print("\n$", " ".join(args))
    proc = subprocess.Popen(
        args,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args)


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + '\n',
        encoding='utf-8',
    )
    tmp.replace(path)


def locate_v1_run(root, tag):
    candidates = [
        root / 'Market_Model_V2' / 'historical_quant_2017_v1' / tag,
        root / 'Kalman' / 'Market_Model_V2' / 'historical_quant_2017_v1' / tag,
        root / 'kalman' / 'Market_Model_V2' / 'historical_quant_2017_v1' / tag,
    ]
    for p in candidates:
        if (p / 'historical_resume_complete.json').exists():
            return p
    raise FileNotFoundError(f'V1 run not found: {tag}')


def locate_v3_run(root, tag):
    candidates = [
        root / 'Market_Model_V2' / 'historical_quant_2017_v3_candidate' / tag,
        root / 'Kalman' / 'Market_Model_V2' / 'historical_quant_2017_v3_candidate' / tag,
        root / 'kalman' / 'Market_Model_V2' / 'historical_quant_2017_v3_candidate' / tag,
    ]
    for p in candidates:
        summary = p / 'historical_v3_candidate_summary.json'
        if summary.exists():
            payload = json.loads(summary.read_text(encoding='utf-8'))
            if payload.get('status') == 'COMPLETE':
                return p
    raise FileNotFoundError(f'COMPLETE V3 run not found: {tag}')


repo = Path('/content/Codex')
if repo.exists():
    shutil.rmtree(repo)

v1_root = locate_v1_run(DRIVE_ROOT, V1_RUN_TAG)
matrix_dir = v1_root.parents[1] / 'historical_matrices_v1'
for market in ('us', 'kr', 'btc'):
    assert (matrix_dir / f'{market}_matrix.parquet').exists()
    assert (matrix_dir / f'{market}_anchor_prices.parquet').exists()

v3_root = locate_v3_run(DRIVE_ROOT, V3_CANDIDATE_TAG)
model_root = v1_root.parents[1]
out_root = (
    model_root
    / 'historical_quant_2017_v3_2_validation'
    / V32_TAG
)
out_root.mkdir(parents=True, exist_ok=True)
status_path = out_root / 'v3_2_colab_status.json'


def status(state, phase, **extra):
    write_json(status_path, {
        'status': state,
        'phase': phase,
        'updated_at': datetime.now().astimezone().isoformat(),
        'pinned_sha': PINNED_SHA,
        'source_v3_tag': V3_CANDIDATE_TAG,
        'v3_2_tag': V32_TAG,
        'research_only': True,
        'live_execution': False,
        'toss_execution': False,
        'neon_write': False,
        **extra,
    })


try:
    status('RUNNING', 'CLONE')
    run([
        'git', 'clone', '--branch', SOURCE_BRANCH,
        'https://github.com/kimtk94/Codex.git', repo,
    ])
    run(['git', '-C', repo, 'checkout', '--detach', PINNED_SHA])
    checked = subprocess.check_output(
        ['git', '-C', repo, 'rev-parse', 'HEAD'],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / 'kalman-toss-gateway'
    module = (
        app / 'research' / 'quant_stack'
        / 'historical_v3_2_portfolio_validation.py'
    )
    test_file = (
        app / 'tests'
        / 'test_historical_v3_2_portfolio_validation.py'
    )
    spec = app / 'config' / 'model-v3-historical-return-regime-spec.json'
    for required in (module, test_file, spec):
        assert required.exists(), required

    status('RUNNING', 'ISOLATED_ENV')
    if shutil.which('uv') is None:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    uv = shutil.which('uv')
    assert uv, 'uv not found'

    venv = Path('/content/.venv-kalman-v3-2')
    if venv.exists():
        shutil.rmtree(venv)
    run([uv, 'venv', venv])
    vpy = venv / 'bin' / 'python'

    run([
        uv, 'pip', 'install', '--python', vpy,
        'pandas>=3.0.3',
        'numpy>=2.4.6',
        'pyarrow',
        'scipy<1.18',
        'scikit-learn',
        'PyPortfolioOpt==1.6.0',
        'riskfolio-lib==7.3.0',
        'vectorbt==1.1.0',
        'plotly<7',
        'pytest',
    ])
    run([uv, 'pip', 'check', '--python', vpy])

    status('RUNNING', 'STATIC_AND_UNIT_TESTS')
    run([vpy, '-m', 'py_compile', module, test_file])
    run([
        vpy, '-m', 'pytest', '-q',
        'tests/test_historical_v3_2_portfolio_validation.py',
    ], cwd=app)

    status('RUNNING', 'V3_2_VALIDATION')
    run([
        vpy, '-m',
        'research.quant_stack.historical_v3_2_portfolio_validation',
        '--v3-root', v3_root,
        '--matrix-dir', matrix_dir,
        '--spec', spec,
        '--output-dir', out_root,
        '--code-sha', PINNED_SHA,
        '--lookback-days', '180',
        '--min-observations', '90',
        '--rebalance', 'M',
        '--annualization-days', '365',
        '--rebalance-cost-bps', '10',
    ], cwd=app)

    summary = out_root / 'v3_2_validation_summary.json'
    comparison = out_root / 'v3_2_tournament_comparison.csv'
    ranking = out_root / 'v3_2_eligible_ranking.csv'
    for result in (summary, comparison, ranking):
        assert result.exists(), result

    snap = out_root / 'code_snapshot'
    snap.mkdir(exist_ok=True)
    shutil.copy2(module, snap / module.name)
    shutil.copy2(test_file, snap / test_file.name)
    shutil.copy2(spec, snap / spec.name)

    payload = json.loads(summary.read_text(encoding='utf-8'))
    status(
        'COMPLETE',
        'DONE',
        experiment_status=payload.get('experiment_status'),
        summary=str(summary),
        comparison=str(comparison),
        ranking=str(ranking),
    )

    print('\n' + '=' * 88)
    print('KALMAN V3.2 PORTFOLIO VALIDATION COMPLETE')
    print('=' * 88)
    print(summary.read_text(encoding='utf-8'))
    print('\nELIGIBLE RANKING')
    print(ranking.read_text(encoding='utf-8'))

except Exception as exc:
    status(
        'FAIL',
        'FAILED',
        error_type=type(exc).__name__,
        error=str(exc),
        traceback=traceback.format_exc(),
    )
    raise
